## 2 Parsing of video metadata after collection

**Author:** Kristina Aleksandrovna Pedersen  

In [1]:
#Load relevant modules
#from functions import *
import pandas as pd
from glob import glob
import re
import os
from tqdm import tqdm
from IPython.display import clear_output

In [5]:
#Function for parsing
def clean_dataframe(filepath, destination, cutoff_date = "01-09-2024", cutoff_duration = 40):
    """Parse through a dataframe of tiktok data and asjust for relevant time window and video duration 
    and save file. Note: saves the dataframe directly, and does not return a dataframe object.
    filepath: path to dataframe
    destination: destination path of cleaned dataframe
    cutoff_date: (str) earliest observations to keep (any older videos are excluded).
    cutoff_duration: (int) maximum number of seconds of videos to be kept."""
    
    import pandas as pd
    from datetime import datetime
    import pytz

    #load dataframe
    df = pd.read_pickle(filepath)
    #transform the timestamp column into a datetime column
    df['video_datetime'] = df.video_timestamp.apply(lambda x: datetime.fromtimestamp(int(x), pytz.utc) if pd.notnull(x) else None)

    #try so that error is not thrown on empty dataframes - if this occurs the function does/returns nothing
    try:
        #convert new datetime column to a date
        df['video_date'] = df.video_datetime.dt.date
        #save cutoff date as datetime date
        cutoff_date = pd.to_datetime(cutoff_date, utc = True, dayfirst = True).date()
        #only keep observations after the cutoff date and shorter than the maximum number of seconds set
        df = df[(df.video_date >= cutoff_date) & (df.video_dur_secs <= cutoff_duration)].reset_index(drop = True)

        #save the dataframe in new destination only if any observations are left
        if not df.empty:
            df.to_pickle(destination)
    except:
        pass

#### Political videos

In [1]:
#referring back to main folder containing video metadata. File structure here is individual partyfolders with
#user level dataframes.
party_folders = glob("../video_metadata/*")

#Loop through each party folder
for party_folder in tqdm(party_folders):
    clear_output(wait = True)

    #Isolating party name
    party_cat = re.search("../video_metadata/(.*)", party_folder).group(1)
    #Specifying contents of partyfolder
    party_folder = glob(f"{party_folder}/*")
    
    #make or define folder for cleaned party metadata
    destination_folder = f"../video_metadata_clean/{party_cat}/"  # Replace with your destination folder path
    # Create the destination folder if it doesn't exist
    os.makedirs(destination_folder, exist_ok=True)

    #Loop through each user level dataframe
    for user_file in tqdm(party_folder):
        clear_output(wait = True)

        #extract user handle (/filename)
        username = os.path.split(user_file)[1]
        #define new destination path
        new_destination = f"{destination_folder}/{username}"

        #apply cleaning function to the metadata dataframe
        clean_dataframe(user_file, new_destination, 
                        cutoff_date = '01-09-2024') 

### Entertainment

In [2]:
#Apply the function (all video metadata stored in one dataframe)
clean_dataframe("entertainment/ent_video_metadata.pkl", 
                "entertainment/ent_video_metadata_clean.pkl", 
                cutoff_date = "01-01-2024", #note the longer timespan allowed
                cutoff_duration = 20) #note the shorter video length cutoff (20 seconds)